# Cardiac Sub-Structure Segmentation Example

> Important: When running on Google Colab, ensure you change your runtime to use a GPU before
proceeding (Runtime->Change runtime type, select GPU as Hardware accelerator and press Save)

## Import Modules

In [1]:
try:
    import platipy
except:
    # We install platipy with the 'cardiac' extra since that contains some extra libraries we need.
    !pip install platipy[cardiac]
    import platipy

from matplotlib import pyplot as plt

import SimpleITK as sitk

from platipy.imaging.tests.data import get_lung_nifti
from platipy.imaging.projects.cardiac.run import run_hybrid_segmentation
from platipy.imaging import ImageVisualiser
from platipy.imaging.label.utils import get_com

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.6/276.6 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of dicom2nifti to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.0 MB/s eta 0:00:00
  Created wheel for n

In [2]:
import torch

# 1. Guardamos a função original do PyTorch
_original_torch_load = torch.load

# 2. Criamos uma versão "modificada" que desliga a trava de segurança automaticamente
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

# 3. Injetamos a versão modificada de volta no PyTorch
torch.load = _patched_torch_load

print("✅ Trava de segurança do PyTorch 2.6 desligada com sucesso!")

✅ Trava de segurança do PyTorch 2.6 desligada com sucesso!


## Download Test Data

This will download lung cancer patient CT scans, with contours of several structures.
This only has to be done once - if it is ran again don't worry, the same data will not be downloaded again!

In [3]:
data_path = get_lung_nifti()

## Load Test Image

Read in the image we want to automatically segment


In [4]:
test_pat_path = data_path.joinpath("LCTSC-Test-S1-201")
test_image = sitk.ReadImage(str(test_pat_path.joinpath("IMAGES/LCTSC_TEST_S1_201_0_CT_0.nii.gz")))

## Run Auto-segmentation

This will take some time, and will print updates along the way


In [5]:
auto_structures, _ = run_hybrid_segmentation(test_image)



Please cite the following paper when using nnUNet:

Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. "nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation." Nat Methods (2020). https://doi.org/10.1038/s41592-020-01008-z


If you have questions or suggestions, feel free to open an issue at https://github.com/MIC-DKFZ/nnUNet

/root/.platipy/nnUNet_models/nnUNet/3d_lowres/Task400_OPEN_HEART_1FOLD
This model expects 1 input modalities for each image
Found 1 unique case ids, here are some examples: ['Task400_OPEN_HEART_1FOLD']
If they don't look right, make sure to double check your filenames. They must end with _0000.nii.gz etc
emptying cuda cache
loading parameters for folds, all
using the following model files:  ['/root/.platipy/nnUNet_models/nnUNet/3d_lowres/Task400_OPEN_HEART_1FOLD/nnUNetTrainerV2__nnUNetPlansv2.1/all/model_final_checkpoint.model']
starting preprocessing generator
starting prediction...
preprocessing /tmp/tmp66tsants/Task400_OPEN_HEA

/usr/local/lib/python3.12/dist-packages/nnunet/training/network_training/network_trainer.py:404: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.amp_grad_scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/nnunet/network_architecture/neural_network.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with context():


done
prediction done
inference done. Now waiting for the segmentation export to finish...
force_separate_z: None interpolation order: 1
separate z: True lowres axis [0]
separate z, order in z is 0 order inplane is 1
WARNING! Cannot run postprocessing because the postprocessing file is missing. Make sure to run consolidate_folds in the output folder of the model first!
The folder you need to run this in is /root/.platipy/nnUNet_models/nnUNet/3d_lowres/Task400_OPEN_HEART_1FOLD/nnUNetTrainerV2__nnUNetPlansv2.1


## Save segmentations
Optionally write the automatic segmentations to disk


In [6]:
output_directory = test_pat_path.joinpath("substructures")
output_directory.mkdir(exist_ok=True)

for struct_name in list(auto_structures.keys()):
    sitk.WriteImage(auto_structures[struct_name], str(output_directory.joinpath(f"{struct_name}.nii.gz")))

print(f"Segmentations saved to: {output_directory}")

Segmentations saved to: data/nifti/lung/LCTSC-Test-S1-201/substructures


## Visualise Segmentations

Next, we can generate a nice figure to check what the segmentation looks like using platipy's ImageVisualiser

In [7]:
vis = ImageVisualiser(test_image, cut=get_com(auto_structures["Heart"]))
vis.add_contour({struct: auto_structures[struct] for struct in auto_structures.keys()})
vis.set_limits_from_label(auto_structures["Heart"], expansion=20)
vis.show()
snapshot_path = output_directory.joinpath(f"snapshot.png")
plt.savefig(snapshot_path)
print(f"Snapshot saved to: {snapshot_path}")

Snapshot saved to: data/nifti/lung/LCTSC-Test-S1-201/substructures/snapshot.png
